# Step 2c — Unified-label loader (ScienceQA scheme everywhere)



In [ ]:
!pip install -q datasets openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['PROJECT_ROOT'] = '/content/drive/MyDrive/text-difficulty-classification'
print('PROJECT_ROOT:', os.environ['PROJECT_ROOT'])
print('train.csv exists:', os.path.exists(f"{os.environ['PROJECT_ROOT']}/data/train.csv"))

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/text-difficulty-classification
train.csv exists: True


In [ ]:
# Mount Drive + set PROJECT_ROOT
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/text-difficulty-classification'
except Exception:
    PROJECT_ROOT = os.path.abspath('.')
os.environ['PROJECT_ROOT'] = PROJECT_ROOT
print('PROJECT_ROOT =', PROJECT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT = /content/drive/MyDrive/text-difficulty-classification


In [ ]:
import argparse
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = os.environ['PROJECT_ROOT']
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data')
MULTI_DIR    = os.path.join(PROJECT_ROOT, 'outputs', 'multi_corpus')
os.makedirs(MULTI_DIR, exist_ok=True)

LABEL_NAMES = ('elementary', 'middle', 'high')
RANDOM_SEED = 42


# ---------- canonical mapping (ScienceQA convention) ----------


In [ ]:
def grade_to_level(grade):
    """1-5 -> elementary, 6-8 -> middle, 9-12 -> high. Returns None if out
    of range or unparseable."""
    try:
        g = float(grade)
    except (TypeError, ValueError):
        return None
    if pd.isna(g): return None
    g_int = int(round(g))
    if 1  <= g_int <= 5:  return 'elementary'
    if 6  <= g_int <= 8:  return 'middle'
    if 9  <= g_int <= 12: return 'high'
    if g_int > 12:        return 'high'    # college texts → cap at 'high'
    return None


In [ ]:
def _validate_labels(df, source):
    """Hard-fail if any label is outside the canonical 3-set."""
    bad = df[~df['education_level'].isin(LABEL_NAMES)]
    if len(bad):
        sys.exit(f"[step2c] {source}: {len(bad)} rows with non-canonical labels: "
                 f"{bad['education_level'].unique().tolist()[:5]}")
    return df


In [ ]:
# ---------- per-corpus loaders ----------


In [ ]:
def load_scienceqa():
    """Reuse Part-1 train/test CSVs. Already grade-1-12 mapped."""
    paths = [(os.path.join(DATA_DIR, 'train.csv'), 'train'),
             (os.path.join(DATA_DIR, 'test.csv'),  'test')]
    if not all(os.path.exists(p) for p, _ in paths):
        sys.exit("[step2c] ScienceQA train/test CSVs missing. Run Part 1.")
    dfs = []
    for path, _ in paths:
        d = pd.read_csv(path)
        dfs.append(d)
    df = pd.concat(dfs, ignore_index=True)
    df['source_dataset'] = 'scienceqa'
    df['domain']         = df.get('subject', pd.Series(['science']*len(df))).fillna('science')
    df['label_source']   = 'grade'
    df['subject']        = df.get('subject', pd.Series([''] * len(df))).fillna('')
    df['raw_label']      = df.get('grade', pd.Series([''] * len(df))).astype(str)
    return _validate_labels(
        df[['full_text', 'education_level', 'source_dataset',
            'domain', 'label_source', 'subject', 'raw_label']],
        'scienceqa')


In [ ]:
def load_onestop():
    """SetFit/onestop_english. 3 levels mapped by typical CEFR grade band:
        Elementary  ≈ B1   ≈ ~grade 4   -> elementary
        Intermediate ≈ B2  ≈ ~grade 7   -> middle
        Advanced    ≈ C1   ≈ ~grade 11  -> high
    """
    from datasets import load_dataset
    ds = load_dataset('SetFit/onestop_english')
    rows = []
    for split in ds:
        for r in ds[split]:
            label_int = int(r.get('label', -1))

            mapping = {0: 'elementary', 1: 'middle', 2: 'high'}
            level = mapping.get(label_int)
            if level is None:
                continue
            text = r.get('text') or r.get('content') or ''
            if text.strip():
                rows.append({
                    'full_text': text, 'education_level': level,
                    'source_dataset': 'onestop', 'domain': 'reading',
                    'label_source': 'grade_band', 'subject': 'news',
                    'raw_label': str(label_int),
                })
    return _validate_labels(pd.DataFrame(rows), 'onestop')


In [ ]:
def load_clear(xlsx_path):
    """CLEAR xlsx — uses Flesch-Kincaid-Grade-Level column for the grade
    integer. Maps 1:1 to ScienceQA's grade scheme."""
    if not xlsx_path or not os.path.exists(xlsx_path):
        sys.exit(f"[step2c] CLEAR xlsx not found at {xlsx_path}. "
                 "Pass --clear-path /path/to/CLEAR_corpus_final.xlsx")
    df = pd.read_excel(xlsx_path)
    grade_col = next((c for c in df.columns
                      if 'flesch-kincaid-grade-level' in c.lower().replace(' ', '-')
                      or c.strip().lower() == 'flesch-kincaid-grade-level'), None)
    text_col = next((c for c in df.columns if c.lower().strip() == 'excerpt'), None)
    if grade_col is None or text_col is None:
        sys.exit(f"[step2c] CLEAR: missing FK grade or Excerpt column. "
                 f"Found: {list(df.columns)[:30]}")
    df = df.rename(columns={grade_col: '_fk_grade', text_col: 'full_text'})
    df['education_level'] = df['_fk_grade'].apply(grade_to_level)
    df = df.dropna(subset=['education_level', 'full_text']).reset_index(drop=True)
    df['source_dataset'] = 'clear'
    df['domain']         = df.get('Categ', pd.Series(['reading']*len(df))).fillna('reading')
    df['label_source']   = 'grade'    # FK-Grade is a grade integer
    df['subject']        = df.get('Sub Cat', pd.Series([''] * len(df))).fillna('')
    df['raw_label']      = df['_fk_grade'].astype(str)
    return _validate_labels(
        df[['full_text', 'education_level', 'source_dataset',
            'domain', 'label_source', 'subject', 'raw_label']],
        'clear')


In [ ]:
def load_race(config):
    """ehovy/race. config='middle' (gr 6-9 → middle) or 'high' (gr 9-12 → high)."""
    if config not in ('middle', 'high'):
        sys.exit(f"[step2c] race config must be middle|high, got {config}")
    from datasets import load_dataset
    ds = load_dataset('ehovy/race', config)
    level = 'middle' if config == 'middle' else 'high'
    rows = []
    seen = set()
    for split in ds:
        for r in ds[split]:
            article = r.get('article', '')
            if not article.strip() or article in seen:
                continue
            seen.add(article)
            rows.append({
                'full_text': article, 'education_level': level,
                'source_dataset': f'race-{config}', 'domain': 'exam',
                'label_source': 'dataset_level', 'subject': 'reading-comp',
                'raw_label': config,
            })
    return _validate_labels(pd.DataFrame(rows), f'race-{config}')


In [ ]:
def load_mctest():
    """sagnikrayc/mctest — children's stories grades 1-4 (within elementary band)."""
    from datasets import load_dataset
    rows, seen = [], set()
    ds = load_dataset('sagnikrayc/mctest', 'mc500')
    for split in ds:
        for r in ds[split]:
            story = r.get('story') or r.get('text') or ''
            if not story.strip() or story in seen:
                continue
            seen.add(story)
            rows.append({
                'full_text': story, 'education_level': 'elementary',
                'source_dataset': 'mctest', 'domain': 'fiction',
                'label_source': 'grade_band', 'subject': 'children-story',
                'raw_label': 'gr1-4',
            })
    return _validate_labels(pd.DataFrame(rows), 'mctest')


In [ ]:
def load_openbookqa():
    """allenai/openbookqa — author-stated grade-4 elementary science."""
    from datasets import load_dataset
    ds = load_dataset('allenai/openbookqa', 'main')
    rows = []
    for split in ds:
        for r in ds[split]:
            q = r.get('question_stem') or r.get('question', '')
            if isinstance(q, dict):
                q = q.get('stem', '')
            choices = r.get('choices', {})
            ctxt = ' '.join(choices.get('text', [])) if isinstance(choices, dict) else ''
            text = (q + ' ' + ctxt).strip()
            if text:
                rows.append({
                    'full_text': text, 'education_level': 'elementary',
                    'source_dataset': 'openbookqa', 'domain': 'science',
                    'label_source': 'grade_band', 'subject': 'science-qa',
                    'raw_label': 'gr4',
                })
    return _validate_labels(pd.DataFrame(rows), 'openbookqa')


In [ ]:
LOADERS = {
    'scienceqa':   lambda **_: load_scienceqa(),
    'onestop':     lambda **_: load_onestop(),
    'clear':       lambda clear_path=None, **_: load_clear(clear_path),
    'race-middle': lambda **_: load_race('middle'),
    'race-high':   lambda **_: load_race('high'),
    'mctest':      lambda **_: load_mctest(),
    'openbookqa':  lambda **_: load_openbookqa(),
}


# ---------- balance + split ----------


In [ ]:
def _filter_strict(df):
    """Drop rows where label_source != 'grade' (only keeps grade-integer derived)."""
    return df[df['label_source'] == 'grade'].reset_index(drop=True)


In [ ]:
def _balance(df, max_per_level):
    if max_per_level is None:
        return df
    out = []
    for (_, _), g in df.groupby(['source_dataset', 'education_level']):
        out.append(g.sample(n=min(len(g), max_per_level), random_state=RANDOM_SEED))
    return pd.concat(out, ignore_index=True)


In [ ]:
def _stratified_split(df, val_frac=0.1, test_frac=0.1):
    rng = np.random.default_rng(RANDOM_SEED)
    train, val, test = [], [], []
    for (_, _), g in df.groupby(['source_dataset', 'education_level']):
        idx = rng.permutation(len(g))
        n = len(idx)
        n_val  = max(1, int(n * val_frac))
        n_test = max(1, int(n * test_frac))
        test.append(g.iloc[idx[:n_test]])
        val.append(g.iloc[idx[n_test:n_test + n_val]])
        train.append(g.iloc[idx[n_test + n_val:]])
    return {'train': pd.concat(train, ignore_index=True),
            'val':   pd.concat(val,   ignore_index=True),
            'test':  pd.concat(test,  ignore_index=True)}


In [ ]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--include', nargs='+', default=['scienceqa'],
                    choices=list(LOADERS),
                    help='In-distribution corpora (split into train/val/test).')
    ap.add_argument('--ood', nargs='+', default=[], choices=list(LOADERS),
                    help='OOD corpora (held out as ood_<source>.csv).')
    ap.add_argument('--clear-path', default=None,
                    help='Local path to CLEAR_corpus_final.xlsx.')
    ap.add_argument('--strict', action='store_true',
                    help='Drop rows whose label_source != grade (filters out grade_band).')
    ap.add_argument('--max-per-level', type=int, default=None,
                    help='Cap rows per (source, level).')
    args = ap.parse_args()

    overlap = set(args.include) & set(args.ood)
    if overlap:
        sys.exit(f"[step2c] {overlap} listed in both --include and --ood.")

    print(f"[step2c] include={args.include} ood={args.ood} strict={args.strict}")

    # Build in-distribution
    in_dfs = []
    for src in args.include:
        print(f"  loading IN-dist: {src}")
        d = LOADERS[src](clear_path=args.clear_path)
        in_dfs.append(d)
    in_df = pd.concat(in_dfs, ignore_index=True)
    if args.strict:
        before = len(in_df)
        in_df = _filter_strict(in_df)
        print(f"  --strict: kept {len(in_df)}/{before} grade-derived rows")
    in_df = _balance(in_df, args.max_per_level)
    in_df = _validate_labels(in_df, 'in-distribution')
    print(f"  in-dist total: {len(in_df)}  per-level: "
          f"{in_df['education_level'].value_counts().to_dict()}")

    splits = _stratified_split(in_df)
    for name, df in splits.items():
        df = df.copy()
        df['split'] = name
        out = os.path.join(MULTI_DIR, f'{name}.csv')
        df.to_csv(out, index=False)
        print(f"  wrote {out}  rows={len(df)}")

    for src in args.ood:
        print(f"  loading OOD: {src}")
        d = LOADERS[src](clear_path=args.clear_path)
        if args.strict:
            d = _filter_strict(d)
        d = _validate_labels(d, src)
        d['split'] = 'ood_test'
        out = os.path.join(MULTI_DIR, f'ood_{src}.csv')
        d.to_csv(out, index=False)
        print(f"  wrote {out}  rows={len(d)}  per-level: "
              f"{d['education_level'].value_counts().to_dict()}")

    print(f"\n[step2c] done — files in {MULTI_DIR}")


In [ ]:
import sys
sys.argv = ['step2c',
    '--include', 'scienceqa', 'clear',
    '--ood', 'onestop', 'race-middle', 'race-high', 'openbookqa',
    '--clear-path',
    '/content/drive/MyDrive/text-difficulty-classification/data/raw/CLEAR_corpus_final.xlsx',
    '--max-per-level', '1500']
main()


[step2c] include=['scienceqa', 'clear'] ood=['onestop', 'race-middle', 'race-high', 'openbookqa'] strict=False
  loading IN-dist: scienceqa
  loading IN-dist: clear
  in-dist total: 7961  per-level: {'high': 3000, 'middle': 2633, 'elementary': 2328}
  wrote /content/drive/MyDrive/text-difficulty-classification/outputs/multi_corpus/train.csv  rows=6371
  wrote /content/drive/MyDrive/text-difficulty-classification/outputs/multi_corpus/val.csv  rows=795
  wrote /content/drive/MyDrive/text-difficulty-classification/outputs/multi_corpus/test.csv  rows=795
  loading OOD: onestop


README.md:   0%|          | 0.00/439 [00:00<?, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/192 [00:00<?, ? examples/s]

...

main/test-00000-of-00001.parquet:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4957 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

  wrote /content/drive/MyDrive/text-difficulty-classification/outputs/multi_corpus/ood_openbookqa.csv  rows=5957  per-level: {'elementary': 5957}

[step2c] done — files in /content/drive/MyDrive/text-difficulty-classification/outputs/multi_corpus


In [ ]:
import pandas as pd, os
PROJECT_ROOT = '/content/drive/MyDrive/text-difficulty-classification'
MULTI = f'{PROJECT_ROOT}/outputs/multi_corpus'
SEED = 42

def cap(name, n=1500):
    p = f'{MULTI}/{name}.csv'
    df = pd.read_csv(p)
    if len(df) <= n:
        print(f'  {name}: {len(df)} rows (no cap)')
        return
    out = df.groupby('education_level', group_keys=False).apply(
        lambda g: g.sample(n=min(len(g), max(1, n * len(g) // len(df))), random_state=SEED))
    out.to_csv(p, index=False)
    print(f'  {name}: {len(df)} → {len(out)}')

for c in ['ood_race-middle', 'ood_race-high', 'ood_openbookqa']:
    cap(c, 1500)

/tmp/ipykernel_3602/3424848940.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = df.groupby('education_level', group_keys=False).apply(


  ood_race-middle: 7139 → 1500


/tmp/ipykernel_3602/3424848940.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = df.groupby('education_level', group_keys=False).apply(
/tmp/ipykernel_3602/3424848940.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = df.groupby('education_level', group_keys=False).apply(


  ood_race-high: 20792 → 1500
  ood_openbookqa: 5957 → 1500
